# 🐦‍🔥 Fenix Core — train the real adapter

This replaces the old run that produced six steps and a loss of 3.23: an adapter that had a filename, a loss, and no behaviour.

Three things are different here.

1. **It refuses a dataset too small to learn from.** Below the floor it stops and tells you why, instead of training anyway.
2. **It holds out a split** and scores the result on it. A run that cannot beat the untuned base is reported as failed.
3. **It pushes to Hugging Face** so the server can actually serve it, and tells you the one line to set afterwards.

### Before you run

- Runtime → Change runtime type → **T4 GPU** → Save.
- In the next cell, set `FENIX_URL` and paste your Fenix account token so the notebook can fetch your own training set.
- The token is the same one the Fenix app stores. Get it from the browser console: `localStorage.getItem('fenix_token')`.

To get a token, sign in to Fenix on the web first. The dataset is per account — the notebook can only ever fetch your own.

In [ ]:
# --- 0. settings ---------------------------------------------------------
import os, sys, json

FENIX_URL   = "https://YOUR-SERVER-HERE"
FENIX_TOKEN = "paste-your-fenix-token-here"
HF_TOKEN    = "paste-your-huggingface-token-here"   # write access, to upload
HF_REPO     = "Hakari66684/fenix-core-lora"

BASE_MODEL  = "Qwen/Qwen3-4B-Instruct-2507"
MIN_ROWS    = 300        # below this, training is theatre
EPOCHS      = 3
LR          = 1e-4
RANK        = 16
ALPHA       = 32

assert FENIX_TOKEN != "paste-your-fenix-token-here", "set your Fenix token first"
print("Fenix :", FENIX_URL)
print("Repo  :", HF_REPO)
print("GPU is checked in the next cell — if it is a CPU you can stop there.")

In [ ]:
# --- 1. is there actually a GPU? ------------------------------------------
try:
    import torch
except ImportError:
    print("Installing torch…")
    !pip install -q torch --index-url https://download.pytorch.org/whl/cu121
    import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime → Change runtime type → T4 GPU → Save, then run all again.\n"
        "Do NOT run this on a CPU: a 4B adapter on CPU takes hours and lands below\n"
        "the fallback chain it was supposed to replace."
    )
name = torch.cuda.get_device_name(0)
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ {name} — {gb:.0f} GB")
print(f"   4-bit LoRA needs roughly 10 GB, so this is {'enough' if gb >= 12 else 'TIGHT'}.")

In [ ]:
# --- 2. fetch the dataset -------------------------------------------------
import urllib.request, urllib.error, json

req = urllib.request.Request(
    FENIX_URL.rstrip('/') + "/api/training-data",
    headers={"Authorization": "Bearer " + FENIX_TOKEN},
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        raw = r.read().decode('utf-8')
        pairs = int(r.headers.get('X-Fenix-Pairs') or 0)
        distinct = int(r.headers.get('X-Fenix-Distinct') or 0)
except urllib.error.HTTPError as e:
    raise SystemExit(f"The server answered HTTP {e.code}. Sign in to Fenix "
                     f"first and check FENIX_URL.")

rows = [json.loads(l) for l in raw.splitlines() if l.strip()]
print(f"pairs: {len(rows)}   distinct replies: {distinct}")

if len(rows) < MIN_ROWS:
    print()
    print("NOT ENOUGH DATA - and that is the correct answer, not an error.")
    print()
    print(f"  You have {len(rows)} pairs. {MIN_ROWS} is the floor.")
    print()
    print("  Training on less produces an adapter that reports a loss, uploads")
    print("  a file, and changes nothing - which is exactly what the current one")
    print("  did. Use Fenix normally for a while so the flywheel fills, then")
    print("  come back. To see how close you are: python api/train_dataset.py")
else:
    with open('/content/fenix-sft.jsonl', 'w', encoding='utf-8') as f:
        f.write(raw)
    print("saved to /content/fenix-sft.jsonl")

In [ ]:
# --- 3. libraries ---------------------------------------------------------
!pip install -q -U \
    "transformers>=4.44" "peft>=0.12" "datasets>=2.20" \
    "trl>=0.9" "bitsandbytes>=0.43" "accelerate>=0.33" "sentencepiece"
import transformers, peft, datasets, trl
print("transformers", transformers.__version__, "| peft", peft.__version__)

In [ ]:
# --- 4. score the untuned base before touching it -------------------------
# A run that cannot beat this number learned nothing, whatever the training
# loss says. Measuring it up front is what makes the verdict at the end real.
import random
random.seed(17)
rows = [json.loads(l) for l in open('/content/fenix-sft.jsonl', encoding='utf-8') if l.strip()]

def split(items, seed=17):
    idx = list(range(len(items))); random.Random(seed).shuffle(idx)
    n_eval = min(40, max(1, len(items) // 10))
    ev = set(idx[:n_eval])
    return ([r for i, r in enumerate(items) if i not in ev],
            [r for i, r in enumerate(items) if i in ev])

train_rows, eval_rows = split(rows)
print(f"train {len(train_rows)} | held out {len(eval_rows)}")
assert not ({r['messages'][1]['content'] for r in train_rows}
            & {r['messages'][1]['content'] for r in eval_rows}), "split leaked"
print("✅ the held-out split shares nothing with training")

In [ ]:
# --- 5. train --------------------------------------------------------------
import torch
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, TrainingArguments
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def to_text(r):
    return tok.apply_chat_template(r['messages'], tokenize=False, add_generation_prompt=False)

train_ds = Dataset.from_list([{'text': to_text(r)} for r in train_rows])
eval_ds  = Dataset.from_list([{'text': to_text(r)} for r in eval_rows])

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, load_in_4bit=True,
    bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16,
    device_map='auto', trust_remote_code=True)
model.config.use_cache = False

peft_cfg = LoraConfig(r=RANK, lora_alpha=ALPHA, lora_dropout=0.05, bias='none',
                       task_type='CAUSAL_LM',
                       target_modules=['q_proj','k_proj','v_proj','o_proj',
                                       'gate_proj','up_proj','down_proj'])

args = TrainingArguments(
    output_dir='/content/out', num_train_epochs=EPOCHS,
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    learning_rate=LR, lr_scheduler_type='cosine', warmup_ratio=0.03,
    logging_steps=10, eval_strategy='epoch', save_strategy='epoch',
    save_total_limit=2, bf16=True, report_to=[], remove_unused_columns=False,
)

trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds,
                      eval_dataset=eval_ds, peft_config=peft_cfg,
                      tokenizer=tok,
                      formatting_func=lambda ex: {'text': ex['text']})
base_metrics = trainer.evaluate()
base_loss = base_metrics.get('eval_loss')
print(f"\n📊 untuned base held-out loss: {base_loss:.4f}\n")
trainer.train()
tuned = trainer.evaluate()
tuned_loss = tuned.get('eval_loss')
print(f"📉 trained held-out loss:     {tuned_loss:.4f}")

In [ ]:
# --- 6. the verdict --------------------------------------------------------
# Ship or do not ship. No exceptions, whatever the training loss looked like.
ACCEPTABLE = 1.6
ok = True
print(f"untuned  {base_loss:.4f}")
print(f"trained  {tuned_loss:.4f}   ({base_loss - tuned_loss:+.4f})")
print()

if tuned_loss >= base_loss:
    ok = False; print("❌ no better than the untuned base — this adapter learned nothing.")
elif tuned_loss > ACCEPTABLE:
    ok = False; print(f"❌ above the acceptable loss ({ACCEPTABLE}) — do not ship this.")
else:
    print(f"✅ learned, and inside the acceptable loss ({ACCEPTABLE}).")

if not ok:
    print("\nDo not upload it. More data is the fix, not more epochs — a bigger\n",
          "epoch count on a thin dataset just overfits it.")
else:
    print("\nRun the next cell to upload it.")

In [ ]:
# --- 7. upload -------------------------------------------------------------
if not ok:
    print("skipped — the run did not pass the verdict above.")
else:
    trainer.model.save_pretrained('/content/adapter')
    tok.save_pretrained('/content/adapter')
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HF_REPO, repo_type='model', exist_ok=True)
    api.upload_folder(folder_path='/content/adapter', repo_id=HF_REPO,
                    repo_type='model')
    print(f"✅ uploaded to {HF_REPO}")
    print()
    print('Next: publish it as an Inference Endpoint or a Space, then set on the')
    print('server:  CORE_HF_URL=<that url>   and confirm  /api/brains  reports')
    print('the core as live rather than unverified.')